# Infra Monitor `LEAGUE_RECORDS.SEED`
Health check for the **SEED schema** (simulated source system). Covers all schema objects: tables, views, stages, file formats, procedures, and load state.

## How to Use
- Click **Run All** to execute every cell top-to-bottom. Or if you know what you are looking for, run individual cell.

In [ ]:
USE DATABASE LEAGUE_RECORDS;

USE SCHEMA SEED;

---
## 1. Object Inventory
All objects registered in the SEED schema, grouped by type.

In [ ]:
SELECT
    TABLE_TYPE AS OBJECT_TYPE,
    TABLE_NAME AS OBJECT_NAME,
    ROW_COUNT,
    ROUND(BYTES / 1024, 0)::INTEGER AS SIZE_KB,
    CREATED,
    LAST_ALTERED,
    COMMENT
FROM LEAGUE_RECORDS.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'SEED'
ORDER BY TABLE_TYPE, TABLE_NAME;

In [ ]:
SHOW STAGES IN SCHEMA LEAGUE_RECORDS.SEED;

In [ ]:
%%sql
SHOW PROCEDURES IN SCHEMA LEAGUE_RECORDS.SEED;

Verify that the base copy into for all 5 source datasets are successful by comparing row counts in original CSV and seed.

In [ ]:
WITH table_counts AS (
    SELECT
        'matches_summary' AS DATASET,
        COUNT(*) AS TABLE_ROWS
    FROM SEED.MATCHES
        UNION ALL
    SELECT 'players_summary', COUNT(*) FROM SEED.PLAYERS
        UNION ALL
    SELECT 'intervals', COUNT(*) FROM SEED.INTERVALS
        UNION ALL
    SELECT 'items_ref', COUNT(*) FROM SEED.ITEMS_REF
        UNION ALL
    SELECT 'champions_ref', COUNT(*) FROM SEED.CHAMPIONS_REF
),
csv_counts AS (
    SELECT
        'matches_summary' AS DATASET,
        COUNT(*) AS CSV_ROWS
    FROM @SEED.UPLOAD_STG (PATTERN => '.*matches_summary.*')
        UNION ALL
    SELECT 'players_summary', COUNT(*)
    FROM @SEED.UPLOAD_STG (PATTERN => '.*players_summary.*')
        UNION ALL
    SELECT 'intervals', COUNT(*)
    FROM @SEED.UPLOAD_STG (PATTERN => '.*intervals.*')
        UNION ALL
    SELECT 'items_ref', COUNT(*)
    FROM @SEED.UPLOAD_STG (PATTERN => '.*items_ref.*')
        UNION ALL
    SELECT 'champions_ref', COUNT(*)
    FROM @SEED.UPLOAD_STG (PATTERN => '.*champions_ref.*')
)
SELECT
    t.DATASET,
    t.TABLE_ROWS,
    c.CSV_ROWS,
    CASE WHEN t.TABLE_ROWS = c.CSV_ROWS THEN 'PASS' ELSE 'MISMATCH' END AS STATUS
FROM table_counts t
JOIN csv_counts c ON t.DATASET = c.DATASET
ORDER BY t.DATASET;

---
## 2. Stage Inspection
Files present in `@SEED.UPLOAD_STG`. All 5 expected CSVs should appear after upload.

In [ ]:
SELECT
    RELATIVE_PATH,
    ROUND(SIZE / 1024, 0)::INTEGER AS FILE_SIZE_KB,
    LAST_MODIFIED,
    FILE_URL
FROM DIRECTORY(@SEED.UPLOAD_STG)
ORDER BY RELATIVE_PATH;

---
## 3. Simulated Daily Load State
Tracks the simulated daily ingestion pointer. `DAYS_REMAINING = 0` means all data has been loaded.

In [ ]:
SELECT
    CURRENT_LOAD_DATE,
    MIN_DATE,
    MAX_DATE,
    LAST_LOADED_AT,
    -- Derived
    DATEDIFF('day', CURRENT_LOAD_DATE, MAX_DATE) AS DAYS_INGESTED,
    DATEDIFF('day', MIN_DATE, CURRENT_LOAD_DATE) AS DAYS_REMAINING
FROM SEED.LOAD_STATE;

In [ ]:
%%sql
SELECT
    GAME_DATE_DAY,
    COUNT(*) AS MATCHES_ON_DATE
FROM SEED._MATCH_DATE_INDEX
GROUP BY GAME_DATE_DAY
ORDER BY GAME_DATE_DAY DESC
LIMIT 10;

---
## 4. Procedure Check
Confirm that `VALIDATE_SEED_UPLOAD` and `SIMULATE_DAILY_LOAD` are registered and callable.

In [ ]:
SELECT
    "name" AS PROCEDURE_NAME,
    "arguments" AS SIGNATURE,
    "description" AS COMMENT
FROM {{seed_procedures_raw}}
WHERE "schema_name" = 'SEED'
ORDER BY "name";

---
## 5. Data Previews
Quick samples to verify schema shape and data integrity.

In [ ]:
%%sql
SELECT * FROM SEED.MATCHES LIMIT 5;

In [ ]:
%%sql
SELECT * FROM SEED.PLAYERS LIMIT 5;

In [ ]:
%%sql
SELECT * FROM SEED.INTERVALS LIMIT 5;

In [ ]:
%%sql
SELECT * FROM SEED.ITEMS_REF LIMIT 5;

In [ ]:
%%sql
SELECT * FROM SEED.CHAMPIONS_REF LIMIT 5;